In [ ]:
import datetime

import colormaps
import matplotlib.pyplot as plt
import os
import numpy as np
import polars as pl
import xarray as xr
from jetutils.anyspell import get_persistent_jet_spells, mask_from_spells_pl, subset_around_onset, get_persistent_spell_times_from_som, get_spells, extend_spells, gb_index, make_daily
from jetutils.clustering import Experiment, RAW_REALSPACE, labels_to_centers
from jetutils.data import *
from jetutils.geospatial import *
from jetutils.definitions import *
from jetutils.jet_finding import *
from jetutils.plots import *
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MaxNLocator
from pathlib import Path
from tqdm import tqdm
import polars.selectors as cs

os.environ["RUST_BACKTRACE"] = "1"

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline

# rest

In [ ]:
# ds_cesm = xr.open_dataset("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/historical/ds.zarr", engine="zarr")
# ds_cesm = standardize(ds_cesm)
# ds_cesm["theta"] = (ds_cesm["t"] * (1000 / ds_cesm["lev"]) ** KAPPA).astype(np.float32)
# ds_cesm = ds_cesm.drop_vars("t")
# dh = DataHandler.from_basepath_and_da("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/historical/results", ds_cesm)
# exp = JetFindingExperiment(dh)
# all_jets_one_df = exp.find_jets()
# props_as_df = exp.props_as_df()

In [ ]:
ds_future = xr.open_dataset("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/ssp370/ds.zarr", engine="zarr")
ds_future = standardize(ds_future)
# ds_future = extract(ds_future, period=(2060, 2071), members=[1, 2, 3, 4])
ds_future["theta"] = (ds_future["t"] * (1000 / ds_future["lev"]) ** KAPPA).astype(np.float32)
ds_future = ds_future.drop_vars("t")
ds_future = ds_future.load()
jets = find_all_jets(ds_future, n_coarsen=1, hole_size=3)

In [ ]:
both_jets = {}
both_paths = {}
for run in ["historical", "ssp370"]:
    ds = xr.open_dataset(f"/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/{run}/ds.zarr", engine="zarr")
    ds = standardize(ds)
    ds = extract(ds, (1970, 1981), members=[1, 2, 3])
    ds["theta"] = (ds["t"] * (1000 / ds["lev"]) ** KAPPA).astype(np.float32)
    ds = ds.drop_vars("t")
    dh = DataHandler.from_basepath_and_da(f"/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/{run}/results", ds)
    exp = JetFindingExperiment(dh)
    all_jets_one_df = exp.find_jets(force=False, n_coarsen=1, smooth_s=5, alignment_thresh=0.55, base_s_thresh=0.55, int_thresh_factor=0.2, hole_size=5)
    if "is_polar" not in all_jets_one_df.columns:
        all_jets_one_df = exp.categorize_jets(None, ["s", "theta"], force=False, n_init=15, init_params="k-means++", mode="month").cast({"time": pl.Datetime("ms")})
    exp.props_as_df()
    
    phat_jets = all_jets_one_df.filter((pl.col("is_polar").mean().over(["member", "time", "jet ID"]) < 0.5) | ((pl.col("is_polar").mean().over(["member", "time", "jet ID"]) > 0.5) & (pl.col("int").mode().first().over(["member", "time", "jet ID"]) > 1.3e8)))

    phat_jets_catd = phat_jets.with_columns(**{"jet ID": (pl.col("is_polar").mean().over(["member", "time", "jet ID"]) > 0.5).cast(pl.UInt32())})
    
    cross_catd_ofile = exp.path.joinpath("cross_catd.parquet")
    if cross_catd_ofile.is_file():
        cross_catd = pl.read_parquet(cross_catd_ofile)
    else:
        cross_catd = track_jets(phat_jets_catd)
        cross_catd = pers_from_cross_catd(cross_catd)
        cross_catd.write_parquet(cross_catd_ofile)


    # phat_jets = all_jets_one_df.filter((pl.col("is_polar").mean().over(["time", "jet ID"]) < 0.5) | ((pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5) & (pl.col("int").mode().first().over(["time", "jet ID"]) > 1.e8)))
    # both_paths[run] = exp.path
    # both_jets[run] = phat_jets.with_columns(**{"jet ID": (pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5).cast(pl.UInt32())})

In [ ]:
cross_all = track_jets(all_jets_one_df)

In [ ]:
cross_all

In [ ]:
pers_from_cross_catd(cross_all)